In [1]:
#!/usr/bin/env python3
import json
from pathlib import Path

INPUT_JSON  = Path("/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Neighbors_Generation/MODULE_ALL_NEIGHBOR_DATA/Module_AllHop_Refilled_domain_level_priors.json")
OUTPUT_JSON = Path("domain_level_conditional_probs_Jan2026.json")

# optional: if you have ko_counts somewhere
KO_COUNTS_JSON = None  # Path("ko_counts.json")

ko_counts = {}
if KO_COUNTS_JSON:
    ko_counts = json.loads(Path(KO_COUNTS_JSON).read_text())

data = json.loads(INPUT_JSON.read_text())
results = []

for ko_i, neighbors in data.items():
    Ni = float(neighbors.get("_count", 0.0) or 0.0)
    if Ni <= 0:
        continue

    for ko_j, Nij in neighbors.items():
        if ko_j == "_count":
            continue
        Nij = float(Nij or 0.0)
        if Nij <= 0:
            continue

        p_j_given_i = Nij / Ni

        results.append({
            "sink": ko_i,              # i
            "source": ko_j,            # j influences i
            "Nij": Nij,
            "Ni": Ni,                  # count for i in this table
            "Nj": float(ko_counts.get(ko_j, 0.0)) if ko_counts else None,
            "Rij": p_j_given_i,        # if you later replace with "reliable" Rij
            "weight": None             # fill later when you normalize per sink
        })

# compute weight per sink: weight = Rij / sum(Rik) over sources k for same sink
by_sink = {}
for e in results:
    by_sink.setdefault(e["sink"], 0.0)
    by_sink[e["sink"]] += (e["Rij"] or 0.0)

for e in results:
    S = by_sink[e["sink"]]
    e["S"] = S
    e["weight"] = (e["Rij"] / S) if S > 0 else 0.0

OUTPUT_JSON.write_text(json.dumps(results, indent=2))
print(f"Wrote {OUTPUT_JSON} with {len(results)} edges")


Wrote domain_level_conditional_probs.json with 23214 edges
